In [1]:
import pandas as pd
from collections import Counter

# Sample Documents
documents = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "The cat and the dog are friends"
]

def compute_tf(documents):
    # 1. Tokenize each document
    # Converting to lowercase and splitting by whitespace
    tokenized_docs = [doc.lower().split() for doc in documents]
    
    # Get all unique words (features) across the corpus
    all_words = sorted(list(set(word for doc in tokenized_docs for word in doc)))
    
    tf_data = []

    # 2. Compute Term Frequency (TF) for each word in each document
    # TF = (Number of times term appears in doc) / (Total number of terms in doc)
    for doc_tokens in tokenized_docs:
        word_counts = Counter(doc_tokens)
        total_words = len(doc_tokens)
        
        # Calculate TF for each unique word in the current document
        doc_tf = {word: (word_counts.get(word, 0) / total_words) for word in all_words}
        tf_data.append(doc_tf)

    # 3. Display TF values in tabular form
    df_tf = pd.DataFrame(tf_data)
    # Adding a column to identify the document index
    df_tf.insert(0, "Document", [f"Doc {i+1}" for i in range(len(documents))])
    
    return df_tf

# Execution
tf_table = compute_tf(documents)
print("Term Frequency (TF) Table:")
print(tf_table.to_string(index=False))


Term Frequency (TF) Table:
Document      and      are      cat      dog  friends      log      mat       on      sat      the
   Doc 1 0.000000 0.000000 0.166667 0.000000 0.000000 0.000000 0.166667 0.166667 0.166667 0.333333
   Doc 2 0.000000 0.000000 0.000000 0.166667 0.000000 0.166667 0.000000 0.166667 0.166667 0.333333
   Doc 3 0.142857 0.142857 0.142857 0.142857 0.142857 0.000000 0.000000 0.000000 0.000000 0.285714


In [2]:
import math
import pandas as pd
from collections import Counter

# Sample Documents
documents = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "The cat and the dog are friends"
]

def analyze_document_importance(documents):
    # 1. Tokenization and Vocabulary
    tokenized_docs = [doc.lower().split() for doc in documents]
    all_words = sorted(list(set(word for doc in tokenized_docs for word in doc)))
    N = len(documents)
    
    # 2. Compute Document Frequency (DF)
    # DF is the number of documents that contain the term
    df_counts = {}
    for word in all_words:
        df_counts[word] = sum(1 for doc in tokenized_docs if word in doc)
    
    # 3. Calculate IDF for each term
    # IDF = log(Total Documents / Number of docs containing the term)
    idf_values = {}
    for word, df in df_counts.items():
        # We add 1 to the denominator to avoid division by zero in larger datasets (optional)
        idf_values[word] = round(math.log(N / df), 4)

    # Convert to DataFrame for display
    idf_df = pd.DataFrame(list(idf_values.items()), columns=['Term', 'IDF']).sort_values(by='IDF', ascending=False)
    
    # 4. Identify High and Low IDF
    high_idf = idf_df[idf_df['IDF'] == idf_df['IDF'].max()]['Term'].tolist()
    low_idf = idf_df[idf_df['IDF'] == idf_df['IDF'].min()]['Term'].tolist()
    
    return idf_df, high_idf, low_idf

# Execution
idf_results, high, low = analyze_document_importance(documents)

print("--- IDF Values for all Terms ---")
print(idf_results.to_string(index=False))

print(f"\nWords with High IDF (Unique/Rare): {high}")
print(f"Words with Low IDF (Common): {low}")

--- IDF Values for all Terms ---
   Term    IDF
    and 1.0986
    are 1.0986
    log 1.0986
friends 1.0986
    mat 1.0986
    dog 0.4055
    cat 0.4055
     on 0.4055
    sat 0.4055
    the 0.0000

Words with High IDF (Unique/Rare): ['and', 'are', 'log', 'friends', 'mat']
Words with Low IDF (Common): ['the']


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample Documents
documents = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "The cat and the dog are friends"
]

def compute_sklearn_tfidf(docs):
    # 1. Initialize TfidfVectorizer
    # We use lowercase=True by default to match our previous experiments
    vectorizer = TfidfVectorizer()

    # 2. Generate TF–IDF matrix
    # This fits the model to the vocabulary and transforms the documents into a matrix
    tfidf_matrix = vectorizer.fit_transform(docs)

    # --- [TASK: Display Vocabulary] ---
    # get_feature_names_out() returns the sorted unique words
    vocabulary = vectorizer.get_feature_names_out()
    
    # --- [TASK: Display IDF Values] ---
    # vectorizer.idf_ stores the computed IDF for each term
    idf_values = dict(zip(vocabulary, vectorizer.idf_))
    
    # --- [TASK: Display TF-IDF Matrix] ---
    # Convert the sparse matrix to a dense format for a readable DataFrame
    df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vocabulary)
    df_tfidf.insert(0, "Document", [f"Doc {i+1}" for i in range(len(docs))])

    return df_tfidf, idf_values, vocabulary

# Execution
tfidf_table, idf_map, vocab = compute_sklearn_tfidf(documents)

print("--- [TASK 1] Vocabulary ---")
print(list(vocab))

print("\n--- [TASK 2] IDF Values ---")
for word, val in idf_map.items():
    print(f"{word:<10} : {val:.4f}")

print("\n--- [TASK 3] TF-IDF Matrix ---")
print(tfidf_table.to_string(index=False))

--- [TASK 1] Vocabulary ---
['and', 'are', 'cat', 'dog', 'friends', 'log', 'mat', 'on', 'sat', 'the']

--- [TASK 2] IDF Values ---
and        : 1.6931
are        : 1.6931
cat        : 1.2877
dog        : 1.2877
friends    : 1.6931
log        : 1.6931
mat        : 1.6931
on         : 1.2877
sat        : 1.2877
the        : 1.0000

--- [TASK 3] TF-IDF Matrix ---
Document      and      are      cat      dog  friends      log      mat       on      sat      the
   Doc 1 0.000000 0.000000 0.374207 0.000000 0.000000 0.000000 0.492038 0.374207 0.374207 0.581211
   Doc 2 0.000000 0.000000 0.000000 0.374207 0.000000 0.492038 0.000000 0.374207 0.374207 0.581211
   Doc 3 0.424396 0.424396 0.322764 0.322764 0.424396 0.000000 0.000000 0.000000 0.000000 0.501310


In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Define the Documents
documents = [
    "The cat sat on the mat",         # D1
    "The cat is on the mat",          # D2
    "The dog sat on the log",         # D3
    "Artificial Intelligence is the future" # D4
]

def analyze_document_similarity(docs):
    # Generate TF-IDF Matrix
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(docs)
    
    # 2. Compute Cosine Similarity
    # cosine_similarity(A, B) computes similarity between rows of the matrix
    sim_d1_d2 = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
    sim_d1_d3 = cosine_similarity(tfidf_matrix[0], tfidf_matrix[2])[0][0]
    sim_d1_d4 = cosine_similarity(tfidf_matrix[0], tfidf_matrix[3])[0][0]
    
    results = {
        "D1 & D2": sim_d1_d2,
        "D1 & D3": sim_d1_d3,
        "D1 & D4": sim_d1_d4
    }
    
    # 3. Identify the most similar pair
    most_similar_pair = max(results, key=results.get)
    
    return results, most_similar_pair

# Execution
similarity_scores, top_pair = analyze_document_similarity(documents)

print("--- [TASK 1] Cosine Similarity Values ---")
for pair, score in similarity_scores.items():
    print(f"{pair}: {score:.4f}")

print(f"\n--- [TASK 2] Most Similar Document Pair ---")
print(f"The most similar pair is {top_pair} with a score of {similarity_scores[top_pair]:.4f}")

--- [TASK 1] Cosine Similarity Values ---
D1 & D2: 0.8151
D1 & D3: 0.5693
D1 & D4: 0.1505

--- [TASK 2] Most Similar Document Pair ---
The most similar pair is D1 & D2 with a score of 0.8151


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Define the Documents
d1 = "The cat sat on the mat"
d2 = "The cat is on the mat"
d3 = "The dog sat on the log"

documents = [d1, d2, d3]

def compute_jaccard(str1, str2):
    # 1. Convert documents into sets of words
    set1 = set(str1.lower().split())
    set2 = set(str2.lower().split())
    
    # Formula: (Intersection) / (Union)
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    return len(intersection) / len(union)

# 2. Compute Jaccard Similarities
jac_d1_d2 = compute_jaccard(d1, d2)
jac_d1_d3 = compute_jaccard(d1, d3)

# 3. Compute Cosine Similarities for Comparison
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)
cos_d1_d2 = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]
cos_d1_d3 = cosine_similarity(tfidf_matrix[0], tfidf_matrix[2])[0][0]

# Display Results
print(f"{'Pair':<10} | {'Jaccard Score':<15} | {'Cosine Score':<15}")
print("-" * 45)
print(f"{'D1 & D2':<10} | {jac_d1_d2:<15.4f} | {cos_d1_d2:<15.4f}")
print(f"{'D1 & D3':<10} | {jac_d1_d3:<15.4f} | {cos_d1_d3:<15.4f}")

Pair       | Jaccard Score   | Cosine Score   
---------------------------------------------
D1 & D2    | 0.6667          | 0.7874         
D1 & D3    | 0.4286          | 0.5989         
